# A Keypoint Metric That Does Not Care How Big You Are

**Topic:** PCK &nbsp;·&nbsp; **Difficulty:** Medium

"Within 10 pixels" means something completely different for a person filling the
frame and one standing 40 metres away. PCK (percentage of correct keypoints)
fixes this by measuring the error relative to a body-size reference, usually the
torso diagonal or head segment.

Compute it both ways on a dataset with a deliberate scale mix, and watch the raw
pixel metric rank the models backwards.

## The data (created in the next cell)

- `gt` `(300, 17, 2)`: 17 keypoints per person.
- `pred_a`, `pred_b` `(300, 17, 2)`: two models.
- `scales` `(300,)`: the person's size in the image, and half the dataset is small.
- Keypoints 5 and 11 are the shoulder and hip used for the torso reference.

## Your task

With **NumPy only**:

1. `pck_pixel(pred, tau)`: the fraction of keypoints within `tau` pixels of
   ground truth, at `tau = 10`.
2. `pck_norm(pred, alpha)`: the fraction within `alpha * torso` where
   `torso = ||gt[:, 5] - gt[:, 11]||` per person, at `alpha = 0.2`.
3. Set `answers` to a dict with `"pck_pixel_a"`, `"pck_pixel_b"`,
   `"pck_norm_a"`, `"pck_norm_b"`, and `"ranking_flips"`: `1.0` if the two
   metrics disagree about which model is better, else `0.0`.

## Evaluation

Compared to the reference to a tolerance of 1e-6, relative with an absolute
floor of 1e-6. The evaluation cell requires the ranking to flip between the two
metrics.

In [ ]:
import numpy as np

rng = np.random.default_rng(3018)
n, k = 300, 17
scales = np.concatenate([np.full(150, 40.0), np.full(150, 220.0)])
base = rng.normal(0, 1, (n, k, 2))
gt = base * scales[:, None, None] * 0.25 + scales[:, None, None]
gt[:, 5] = gt[:, 0] + np.array([0.0, 1.0]) * scales[:, None] * 0.20
gt[:, 11] = gt[:, 0] - np.array([0.0, 1.0]) * scales[:, None] * 0.20

# model A: error proportional to person size (scale-aware)
pred_a = gt + rng.normal(0, 1, (n, k, 2)) * (scales[:, None, None] * 0.035)
# model B: constant pixel error, so it is great on big people and bad on small
pred_b = gt + rng.normal(0, 1, (n, k, 2)) * 5.0
print(f"gt {gt.shape}, scales: 150 small ({scales[0]:.0f}) and "
      f"150 large ({scales[-1]:.0f})")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("pck_pixel_a", [], 0.7829411764705883, -0.13109411630476375, 1e-06),
    ("pck_pixel_b", [], 0.8692156862745098, -0.14553974895542643, 1e-06),
    ("pck_norm_a", [], 0.9211764705882353, -0.15423995952912098, 1e-06),
    ("pck_norm_b", [], 0.5954901960784313, -0.09970769627286939, 1e-06),
    ("ranking_flips", [], 1.0, -0.16743801481449916, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "ranking_flips" in answers:
    _f = float(answers["ranking_flips"]) == 1.0
    print(f"  {'ranking flips':<24} {'ok' if _f else 'wrong'}  "
          f"(pixel: A {answers['pck_pixel_a']:.3f} vs B {answers['pck_pixel_b']:.3f}, "
          f"norm: A {answers['pck_norm_a']:.3f} vs B {answers['pck_norm_b']:.3f})")
    _ok = _ok and _f

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Model B has a fixed 5-pixel error regardless of person size, so on the large
people it looks superb by the pixel metric and drags the average up. Normalizing
by torso length asks the question you actually care about, whether the joint is in the
right place *relative to the body*, and model A, whose error scales with
person, wins. Any keypoint metric quoted in raw pixels is only meaningful on a
dataset with one subject scale.

```python
import numpy as np

err = np.linalg.norm(pred_a - gt, axis=2)
err_b = np.linalg.norm(pred_b - gt, axis=2)

torso = np.linalg.norm(gt[:, 5] - gt[:, 11], axis=1)

def pck_pixel(e, tau=10.0):
    return float((e <= tau).mean())

def pck_norm(e, alpha=0.2):
    return float((e <= alpha * torso[:, None]).mean())

pa, pb = pck_pixel(err), pck_pixel(err_b)
na, nb = pck_norm(err), pck_norm(err_b)

answers = {
    "pck_pixel_a": pa,
    "pck_pixel_b": pb,
    "pck_norm_a": na,
    "pck_norm_b": nb,
    "ranking_flips": float((pa > pb) != (na > nb)),
}
```